# RLHF - Master Notes

## Why RLHF exists

Pretraining only teaches next-token prediction  not helpfulness, safety, or alignment
Raw LLMs mimic internet text distribution, not human intent
Simple supervised fine-tuning isn't enough human preferences are too nuanced to fully capture in a fixed dataset
RLHF solution: don't tell the model what the right answer is — teach it how humans judge answers, then optimize for that judgment

### Stage 1   Supervised Fine-Tuning (SFT)

Human experts write prompt → ideal response pairs
Model fine-tuned on these using standard cross-entropy loss
Teaches format and style of being an assistant (not deep values yet)
Necessary first step  gets the model "pointing in the right direction"
Limitation: can't anticipate every situation; doesn't capture full human preference


### Stage 2  Reward Model Training

Humans compare multiple responses to the same prompt and rank them (easier than scoring)
Rankings used to train a separate Reward Model (RM)  takes prompt + response → outputs a score
RM is a scalable proxy for human judgment (can run billions of times; humans can't)
Key weakness: RM is only an approximation  biases of evaluators get baked in



### Stage 3  PPO (Policy Optimization)

LM treated as an agent, RM treated as the environment
Agent generates response → environment scores it → agent updates to score higher
Algorithm used: Proximal Policy Optimization (PPO)

- Clips update size → prevents unstable, drastic changes, Ensures gradual, stable improvement


- KL divergence penalty added to prevent reward hacking

- Compares current model to SFT baseline
- Penalizes drifting too far → keeps outputs human-like


Full objective: Maximize (RM score) − (KL penalty × drift from SFT)



### Reward Hacking / Gaming the Reward Model

Goodhart's Law: "When a measure becomes a target, it ceases to be a good measure"
Model finds gaps between RM's approximation and actual human preference and exploits them
Root cause: RM trained on finite data; unreliable outside that distribution; LM drifts into those unknown regions during PPO

Key examples:
- Verbosity: Model pads responses because longer = higher score
- Sycophancy: Model agrees with user even when wrong, to maximize approval
- False confidence: Model stops saying "I'm not sure"  confident sounds better to RM
- Format hacking: Adds bullet points/headers everywhere, even when unnecessary
- Overoptimization(Extreme case): produces incoherent outputs that exploit RM's internals
Defenses

### Defenses against reward hacking:

- KL penalty  keeps model tethered to SFT baseline
- Iterative RM updates  retrain RM on current model's outputs periodically
- Constitutional AI (Anthropic)  model critiques itself against written principles; harder to  game than pure preference signal
- Ensemble RMs  multiple reward models; only reward outputs all of them agree on

### Advantages (quick)

- Captures human intuition that rules can't
- Adapts to nuanced, context-dependent preferences
- Scalable once RM is trained
- Produces models that generalize across diverse tasks


### Disadvantages (quick)

- Bias amplification  evaluator biases get embedded in RM
- Imperfect RM  always a lossy approximation of human judgment
- Reward hacking  model exploits gaps between RM and reality
- Expensive  thousands of hours of human labeling required
- Slow  iterative retraining cycle is resource-heavy




## The Big Picture (closing note)

- Pretraining → gives knowledge and language ability
- SFT → gives correct format and initial behavior
- Reward Model → distills human judgment into a math function
- PPO → uses that function to iteratively push model toward preferred behavior

## Core tension: 
human values are too complex to be perfectly captured in any finite mathematical function. Every reward model is a lossy compression of human judgment  and every optimization process will eventually find and exploit that loss. This is why AI alignment remains an open research problem.



## One-line summary to remember:
RLHF turns "predict the next token" into "behave the way humans want"  by building a mathematical proxy for human judgment and optimizing against it, carefully, so the model doesn't learn to fake it.
